In [ ]:
# from selenium import webdriver
# from selenium.webdriver import Chrome, ChromeOptions
# from selenium.webdriver.chrome.service import Service
# from webdriver_manager.chrome import ChromeDriverManager
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.support.ui import  WebDriverWait
# import pandas as pd 
# import time, threading, os, requests, ast

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import json

def scrape(path):
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()
    try:
        # Extract auction details
        auction_name = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/div[@class="auction-header row"]/div[1]/h1'))
        ).text.strip()
        auction_time = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/div[@class="auction-header row"]/div[1]/ul/li[2]'))
        ).text.strip().replace("                ", "")
        auction_center = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/div[@class="auction-header row"]/div[1]/ul/li[1]'))
        ).text.strip().replace("                ", "")

        # Save auction details to JSON file
        auction_data = {
            "auctionname": auction_name,
            "auctiontime": auction_time,
            "auctioncenter": auction_center
        }
        if not os.path.exists("database"):
            os.makedirs("database")
        with open("database/db.json", "w") as f:
            json.dump(auction_data, f, indent=4)
        print("Auction details saved to database/db.json")

        # Rest of your code...
        cars = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/span[@class="sort-page"]'))
        ).text.strip()
        total_cars = int(cars.split(" ")[5])
        print(f"{total_cars} cars found")
    except Exception as e:
        print("Cannot read total cars", e)
        total_cars = 0
    car_count = 0
    while car_count < total_cars:
        try:
            page_cars = WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
            )
            for i in range(len(page_cars)):
                if car_count >= total_cars:
                    break
       
                page_cars = WebDriverWait(driver, 5).until(
                    EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
                )
                driver.execute_script("arguments[0].scrollIntoView();", page_cars[i])
                WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable(page_cars[i])
                ).click()
                try:
                    reg_el = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.XPATH, './/ul[@class="details-list"]/li[1]/strong'))
                    )
                    reg_number = reg_el.text.strip().replace("/", "*").replace(" ", "*")
                    if not os.path.exists("html"):
                        os.makedirs("html")
                    filename = f"html/{reg_number}.html"
                    with open(filename, "w", encoding="utf-8") as f:
                        f.write(driver.page_source)
                    print(f"✔ Saved HTML: {filename}")
                except Exception as e:
                    print("HTML save error", e)
                car_count += 1
                driver.back()
                WebDriverWait(driver, 5).until(
                    EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
                )
            try:
                next_btn = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.XPATH, '//li[@class="page-item page-item-arrow page-item-arrow-next"]/a'))
                )
                next_href = next_btn.get_attribute("href")
                if next_href:
                    print(f"➡ Next page: {next_href}")
                    driver.get(next_href)
                else:
                    print("No more pages")
                    break
            except:
                print("Pagination finished")
                break
        except:
            print("No more car links")
            break
    driver.quit()

# RUN
path = 'https://stock.morrisleslie.com/auction/91'  # replace with the actual URL
scrape(path)

Auction details saved to database/db.json
52 cars found
✔ Saved HTML: html/Margin.html
✔ Saved HTML: html/SR21XOF.html
✔ Saved HTML: html/DT69WYF.html
✔ Saved HTML: html/WV21BBX.html
✔ Saved HTML: html/KV20UYM.html
✔ Saved HTML: html/SV72WUD.html
✔ Saved HTML: html/SH72XEL.html
✔ Saved HTML: html/SD16DWN.html
✔ Saved HTML: html/SK16JYZ.html
✔ Saved HTML: html/ST66KUD.html
✔ Saved HTML: html/HY06WDC.html
✔ Saved HTML: html/SV65KSO.html
✔ Saved HTML: html/SP14YVV.html
✔ Saved HTML: html/KM13BZO.html
✔ Saved HTML: html/EA64CGY.html
✔ Saved HTML: html/SB66EHJ.html
✔ Saved HTML: html/SD65KXA.html
✔ Saved HTML: html/SE16VTK.html
✔ Saved HTML: html/SG25NOF.html
✔ Saved HTML: html/SG58XUK.html
✔ Saved HTML: html/SG63CUX.html
✔ Saved HTML: html/SM59FJU.html
✔ Saved HTML: html/SM62EDF.html
✔ Saved HTML: html/SN64NGO.html
✔ Saved HTML: html/SO16TPX.html
✔ Saved HTML: html/SO65KFY.html
✔ Saved HTML: html/SR16VXX.html
✔ Saved HTML: html/SR18MWJ.html
✔ Saved HTML: html/ST63ZDO.html
✔ Saved HTML: htm

In [5]:
import os,re,json
import csv
import datetime
from bs4 import BeautifulSoup
def details(soup):
    output = {}

    if soup:
        mainDiv = soup.find("ul", class_="details-list")
        if mainDiv:
            items = mainDiv.find_all("li", class_="detail-item")

            for item in items:
                key_tag = item.find("span")
                value_tag = item.find("strong")

                if key_tag and value_tag:
                    key = key_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    output[key] = value

    return output
def extract_image_urls(soup):
    image_urls = []

    if not soup:
        return ""

    thumbs_div = soup.find("div", class_="ug-thumbs-strip")
    if thumbs_div:
        imgs = thumbs_div.find_all("img", class_="ug-thumb-image")
        for img in imgs:
            src = img.get("src", "")
            if src:
                # Remove old prefix
                src = src.replace(
                    "https://dgaww6lqj3.execute-api.eu-west-1.amazonaws.com/prod/buckets/morrisleslie-buckets-s3-public/keys/",
                    ""
                )
                # Remove /resized/
                src = src.replace("/resized/", "/")
                # Remove ---150-100 from filename
                src = src.replace("---150-100", "")
                # Add new prefix
                new_url = f"https://morrisleslie-buckets-s3-public.s3.eu-west-1.amazonaws.com/{src}"
                image_urls.append(new_url)

    # Return comma-separated string
    return ",".join(image_urls)
     

def extract_manual_keys():
    folder = "html"
    output_file = "morrisleslie_data.csv"

    keys = ["Title",
            "Auction Name",
            "Auction type",
            "Center",
            "Make",
            "Model",
            "Variant",
            "Doors",
            "Lot", 
            "Reg", 
            "Start Time", 
            "Start Date",
            "D.O.R",
            "Fuel Type",
            "Former Keepers",
            "Transmission",
            "Colour",
            "MOT Expiry Date",
            "Year",
            "VAT Status",
            "V5",
            # "CAP Clean",
            # "CAP Average",
            # "CAP Below",
            "CC",
            "Mileage",
            "Mileage Warranted",
            # "MOT Expiry Date",
            # "Body Type",
            # "Additional information",
            # "General Condition",
            # "Tyres Condition",
            # "Euro Status",
            # "MOT Due",
            # "Inspection Report",
            "Images",
            # "Damaged_images",
            # "Damage_details",
            ]  

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}
            with open("database/db.json","r",encoding="utf-8") as db:
                dbData = json.load(db)
            row["Auction Name"]=dbData.get("auctionname","")
            row["Center"]=dbData.get("auctioncenter","")
            timeandDate = dbData.get("auctiontime", "")

            if timeandDate:
         
                parts = timeandDate.split("-")
                time_part = parts[0].strip()     
                date_part = parts[1].strip()    


                time_24 = datetime.datetime.strptime(time_part, "%I%p").strftime("%H:%M")

                row["Start Time"] = time_24
                row["Start Date"] = date_part
            else:
                row["Start Time"] = ""
                row["Start Date"] = ""
                
            Make = soup.find("h1", class_="title-h1")
            subs = soup.find_all("p", class_="title-sub")

            make_text = Make.get_text(strip=True) if Make else ""
            model_text = subs[0].get_text(strip=True) if len(subs) > 0 else ""
            variant_full = subs[1].get_text(strip=True) if len(subs) > 1 else ""

       
            doors_match = re.search(r"\b(\d+)dr\b", variant_full.lower())
            doors = doors_match.group(1) if doors_match else ""
            variant_full = re.sub(r"\b\d+dr\b", "", variant_full, flags=re.IGNORECASE).strip()

          
            cc_match = re.search(r"\b(\d+\.\d+|\d{3,4})\b", variant_full)
            cc = cc_match.group(1) if cc_match else ""


            if cc:
                variant_full = re.sub(cc, "", variant_full).strip()


            variant_cleaned = re.sub(r"\s+", " ", variant_full).strip()
            
            lot_tag = soup.find("span", class_="pill-item pill-item-lot")
            if lot_tag:
                lot_text_raw = lot_tag.get_text(strip=True)
            else:
                lot_text_raw = ""   
            lot_lower = lot_text_raw.lower()
            if "tbc" in lot_lower:
                continue      
            lot_num_match = re.search(r"\b(\d+)\b", lot_text_raw)
            if lot_num_match:
                lot = lot_num_match.group(1)  
            else:
                continue  
            
            reg_tag = soup.find("span",class_="pill-item pill-item-reg")
            if reg_tag:
                reg_text =  reg_tag.get_text(strip=True)
           
            else:
                continue
            
 
                
            row["Lot"] = lot
            row["Make"] = make_text
            row["Model"] = model_text
            row["Variant"] = variant_cleaned
            row["Doors"] = doors
            row["CC"] = cc
            row["Title"] = f"{make_text} {model_text}"
            row["Reg"] =  reg_text if reg_text is not None else ""
            get_dteails = details(soup)
            mil = get_dteails.get("Miles", "")
            milage = ""
            if mil:
                milage = "".join(filter(str.isdigit, mil))
            row["D.O.R"] = get_dteails.get("Registered","")
            row["Year"] = get_dteails.get("Year","")
            row["Mileage"] = milage
            row["Mileage Warranted"] = "Yes" if get_dteails.get("Miles Warranted","") == "Warranted" else"No"
            row["Former Keepers"] = get_dteails.get("Former Keepers","")
            row["Fuel Type"] = get_dteails.get("Fuel","")
            row["Transmission"] = get_dteails.get("Transmission","")
            row["Colour"] = get_dteails.get("Colour","")
            row["VAT Status"] = get_dteails.get("VAT","")
            row["MOT Expiry Date"] = get_dteails.get("MOT","")
            row["V5"] = get_dteails.get("V5","")
            row["Auction type"] ="Online Auction"

            images= extract_image_urls(soup)
            row['Images'] = images if images is not None else ""


           
                
            all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=keys)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()



✔ CSV Generated: morrisleslie_data.csv


In [1]:
from urllib.parse import urlparse, urljoin
import threading, requests, os, re
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

df = pd.read_csv("morrisleslie_data.csv")

reg_img = df[['Reg', "Images"]]

def add_watermark_to_image(image_path, text="Sourced from morrisleslie"):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

        try:
            font = ImageFont.truetype("arial.ttf", 50)
        except:
            font = ImageFont.load_default()

        margin = 10
        bbox = draw.textbbox((0, 0), text, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        x, y = image.width - tw - margin, image.height - th - margin

        draw.rectangle([x - margin, y - margin, x + tw + margin, y + th + margin],
                       fill=(0,0,0,160))

        draw.text((x, y), text, font=font, fill=(255,255,255,200))

        watermarked = Image.alpha_composite(image, txt_layer).convert("RGB")
        watermarked.save(image_path)

        print(f"✔ Watermarked: {image_path}")

    except Exception as e:
        print(f"⚠ Watermark Error: {e}")

def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for index, row in data.iterrows():
        reg_no = str(row["Reg"]).strip()
        if not reg_no:
            continue

        img_urls = [u for u in re.split(r',\s*', str(row["Images"])) if u]
        if not img_urls:
            continue

        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)

        def save_img(url, folder, idx):
            url = url.strip()
            if not url:
                return

            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            parsed = urlparse(url)
            if not parsed.netloc:
                print(f"❌ Invalid URL Skipped: {url}")
                return

            full_path = os.path.join(folder, f"{reg_no}_{idx}.jpg")

            if os.path.exists(full_path):
                print(f"⏩ Skipped (Exists): {full_path}")
                return

            try:
                response = requests.get(url, stream=True, timeout=20)
                response.raise_for_status()

                with open(full_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(full_path)
                print(f"📌 Saved: {full_path}")

            except Exception as e:
                print(f"⚠ Error downloading: {url} -> {e}")

        for i, url in enumerate(img_urls):
            save_img(url, reg_folder, i+1)

def start_funcs():
    t1 = threading.Thread(target=download_images, args=(reg_img,))
    t1.start()
    t1.join()

if __name__ == "__main__":
    start_funcs()


✔ Watermarked: Images\DT69WYF\DT69WYF_1.jpg
📌 Saved: Images\DT69WYF\DT69WYF_1.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_2.jpg
📌 Saved: Images\DT69WYF\DT69WYF_2.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_3.jpg
📌 Saved: Images\DT69WYF\DT69WYF_3.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_4.jpg
📌 Saved: Images\DT69WYF\DT69WYF_4.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_5.jpg
📌 Saved: Images\DT69WYF\DT69WYF_5.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_6.jpg
📌 Saved: Images\DT69WYF\DT69WYF_6.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_7.jpg
📌 Saved: Images\DT69WYF\DT69WYF_7.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_8.jpg
📌 Saved: Images\DT69WYF\DT69WYF_8.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_9.jpg
📌 Saved: Images\DT69WYF\DT69WYF_9.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_10.jpg
📌 Saved: Images\DT69WYF\DT69WYF_10.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_11.jpg
📌 Saved: Images\DT69WYF\DT69WYF_11.jpg
✔ Watermarked: Images\DT69WYF\DT69WYF_12.jpg
📌 Saved: Images\DT69WYF\DT69WYF_12.jpg
✔ Watermar